In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SriLankaPortAnalytics").master("local[*]").getOrCreate()

print("Spark is running")
print("Spark version", spark.version)

d:\Big data project\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark is running
Spark version 4.3.0


In [3]:
df = spark.read.csv("../data/raw/lk-port-activity.csv", header=True, inferSchema=True)
df.show(10, truncate=False)
df.distinct().count()

+-------------------+----+-----+---+--------+-----------+---------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+------------+------+----------------+---------------+--------------------+-----------+-------------+------------+------+
|date               |year|month|day|portid  |portname   |country  |ISO3|portcalls_container|portcalls_dry_bulk|portcalls_general_cargo|portcalls_roro|portcalls_tanker|portcalls_cargo|portcalls|import_container|import_dry_bulk|import_general_cargo|import_roro|import_tanker|import_cargo|import|export_container|export_dry_bulk|export_general_cargo|export_roro|export_tanker|export_cargo|export|
+-------------------+----+-----+---+--------+-----------+---------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+--

11412

In [4]:
from pyspark.sql.functions import col, sum, when, trim

missing_values = df.select([
    sum(
        when(
            col(c).isNull() | (trim(col(c).cast("string")) ==""), 1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

missing_values.show()

+----+----+-----+---+------+--------+-------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+------------+------+----------------+---------------+--------------------+-----------+-------------+------------+------+
|date|year|month|day|portid|portname|country|ISO3|portcalls_container|portcalls_dry_bulk|portcalls_general_cargo|portcalls_roro|portcalls_tanker|portcalls_cargo|portcalls|import_container|import_dry_bulk|import_general_cargo|import_roro|import_tanker|import_cargo|import|export_container|export_dry_bulk|export_general_cargo|export_roro|export_tanker|export_cargo|export|
+----+----+-----+---+------+--------+-------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+-----

In [5]:
total_records = df.count()
unique_values = df.dropDuplicates().count()
print("Duplicate records: ", total_records-unique_values)

Duplicate records:  0


In [6]:
for column in df.columns:
    print(f"\nColumn: {column}")
    print("Distinct values:", df.select(column).distinct().count())


Column: date
Distinct values: 2853

Column: year
Distinct values: 8

Column: month
Distinct values: 12

Column: day
Distinct values: 31

Column: portid
Distinct values: 4

Column: portname
Distinct values: 4

Column: country
Distinct values: 1

Column: ISO3
Distinct values: 1

Column: portcalls_container
Distinct values: 17

Column: portcalls_dry_bulk
Distinct values: 6

Column: portcalls_general_cargo
Distinct values: 5

Column: portcalls_roro
Distinct values: 5

Column: portcalls_tanker
Distinct values: 7

Column: portcalls_cargo
Distinct values: 20

Column: portcalls
Distinct values: 22

Column: import_container
Distinct values: 2949

Column: import_dry_bulk
Distinct values: 805

Column: import_general_cargo
Distinct values: 353

Column: import_roro
Distinct values: 267

Column: import_tanker
Distinct values: 749

Column: import_cargo
Distinct values: 3535

Column: import
Distinct values: 3688

Column: export_container
Distinct values: 2908

Column: export_dry_bulk
Distinct values:

In [7]:
df.select("portid", "portname").distinct().show()

+--------+-----------+
|  portid|   portname|
+--------+-----------+
| port678| Hambantota|
|port1319|Trincomalee|
| port254|    Colombo|
|port2224|      Galle|
+--------+-----------+



In [8]:
df.printSchema()

root
 |-- date: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- portid: string (nullable = true)
 |-- portname: string (nullable = true)
 |-- country: string (nullable = true)
 |-- ISO3: string (nullable = true)
 |-- portcalls_container: integer (nullable = true)
 |-- portcalls_dry_bulk: integer (nullable = true)
 |-- portcalls_general_cargo: integer (nullable = true)
 |-- portcalls_roro: integer (nullable = true)
 |-- portcalls_tanker: integer (nullable = true)
 |-- portcalls_cargo: integer (nullable = true)
 |-- portcalls: integer (nullable = true)
 |-- import_container: integer (nullable = true)
 |-- import_dry_bulk: integer (nullable = true)
 |-- import_general_cargo: integer (nullable = true)
 |-- import_roro: integer (nullable = true)
 |-- import_tanker: integer (nullable = true)
 |-- import_cargo: integer (nullable = true)
 |-- import: integer (nullable = true)
 |-- export_container: 

In [9]:
df.select("country", "ISO3").distinct().show()

+---------+----+
|  country|ISO3|
+---------+----+
|Sri Lanka| LKA|
+---------+----+



In [ ]:
df.select("portid", "portname").distinct().show()

+--------+-----------+
|  portid|   portname|
+--------+-----------+
| port678| Hambantota|
|port1319|Trincomalee|
| port254|    Colombo|
|port2224|      Galle|
+--------+-----------+



In [ ]:
df.select("year").distinct().show()

+----+
|year|
+----+
|2025|
|2023|
|2022|
|2026|
|2019|
|2020|
|2024|
|2021|
+----+

